# Solutions · 11 · Continuous-time Markov chains

Worked solutions to the exercises of [notebook 11](../notebooks/11_continuous_time_markov_chains.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engstoch is missing): install it from GitHub
    import engstoch
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engstoch"], check=True)
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engstoch import ctmc, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

R = np.random.default_rng      # R(seed): a fresh, reproducible generator

## Exercise 1

For the machine model, compute the probability that a machine that is up now is down at some point in the next 24 hours (make 'down' absorbing and use P(24)).

In [2]:
Q = models.machine()["Q"].copy()
Qa = Q.copy(); Qa[2] = 0.0                                     # make 'down' absorbing
P24 = ctmc.CTMC(Qa).transition_matrix(24.0)
print(f"P(machine up now is down at some time in the next 24 h) = {P24[0, 2]:.4f}")
print(f"P(down exactly at t = 24 h) = {ctmc.CTMC(Q).transition_matrix(24.0)[0, 2]:.4f} (smaller: it may have been repaired)")
hits = []
for k in range(4000):
    p = models.machine()["chain"].simulate(24.0, "up", R(k)); hits.append(np.any(p["states"] == 2))
print(f"simulation: {np.mean(hits):.4f}")

P(machine up now is down at some time in the next 24 h) = 0.4783
P(down exactly at t = 24 h) = 0.0516 (smaller: it may have been repaired)


simulation: 0.4778


"Ever down within 24 hours" is a first-passage question; making the down state absorbing turns it into "down
at 24 hours" for the modified chain. It is about nine times the probability of being down at the 24-hour mark:
failures are frequent but short, and most are repaired before the mark.

## Exercise 2

Simulate the Lotka-Volterra reaction network from `models.lotka_volterra()` many times and estimate the probability that the predators go extinct before time 50. How does it depend on the initial numbers?

In [3]:
lv = models.lotka_volterra()
rows = []
for x0 in ((50, 100), (100, 50), (20, 20), (200, 20)):
    ext = []
    for k in range(100):
        r = ctmc.gillespie(np.array(x0), lv["stoich"], lv["propensity"], 50.0, R(k), max_events=100_000)
        ext.append(r["x"][-1, 1] == 0)
    rows.append((x0, np.mean(ext)))
print(pd.DataFrame(rows, columns=["(prey, predators) at t = 0", "P(predators extinct by t = 50)"]).to_string(index=False))

(prey, predators) at t = 0  P(predators extinct by t = 50)
                 (50, 100)                            0.20
                 (100, 50)                            0.37
                  (20, 20)                            0.86
                 (200, 20)                            0.82


The deterministic Lotka-Volterra equations cycle for ever; the stochastic system does not - each cycle passes
through a trough of predators where a run of deaths ends the population. Starting with few predators or far
from equilibrium (big oscillations, deep troughs) makes extinction much more likely. Demographic noise is
decisive in small populations, which is why conservation biology uses stochastic models.

## Exercise 3

**Implement it yourself:** write uniformisation from scratch with the truncation chosen from the Poisson tail, and reproduce `CTMC.transition_matrix` for the machine at t = 1, 10 and 100 h.

In [4]:
def uniformise(Q, t, tol=1e-12):
    L = np.max(-np.diag(Q)); Rm = np.eye(len(Q)) + Q / L
    w = np.exp(-L * t); total = w; P = w * np.eye(len(Q)); Rk = np.eye(len(Q)); k = 0
    while 1 - total > tol:
        k += 1; w *= L * t / k; Rk = Rk @ Rm; P += w * Rk; total += w
    return P, k
c = models.machine()["chain"]
for t in (1.0, 10.0, 100.0):
    P, k = uniformise(c.Q, t)
    print(f"t = {t:5.0f} h: {k + 1} terms, max difference from exp(Qt) {np.max(np.abs(P - c.transition_matrix(t))):.1e}")

t =     1 h: 16 terms, max difference from exp(Qt) 2.6e-13
t =    10 h: 44 terms, max difference from exp(Qt) 9.0e-13
t =   100 h: 206 terms, max difference from exp(Qt) 6.4e-13


The number of terms grows like Λt + a few √(Λt): at t = 100 h with Λ = 1.2 per hour about 170 terms. For stiff
chains (a fast rate next to a slow one) and long times uniformisation becomes expensive, while scaling and
squaring does not care; uniformisation's advantages are positivity and an a-priori error bound.